Sentiment Analysis of the IMDB Movies Dataset:

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
import os
import random
from torch.utils.data import random_split
import re
from collections import Counter, OrderedDict
from typing import Dict, List

Task: Reading in the data...

In [2]:
class TextDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.samples = []
        self.transform = transform

        combined_path = os.path.join(root_dir, split)
        for label_name in ['pos','neg']:
            label_dir = os.path.join(combined_path, label_name)
            label = 1 if label_name == 'pos' else 0
            for fname in os.listdir(label_dir):
                if fname.endswith('.txt'):
                    path = os.path.join(label_dir,fname)
                    self.samples.append((path,label))
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        path,label = self.samples[index]
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        if self.transform:
            text = self.transform(text)
        return text,label

In [3]:
root_dir = "/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/data/aclImdb"
train_dataset = TextDataset(root_dir, split='train')
test_dataset = TextDataset(root_dir, split='test')

In [4]:
len(train_dataset), len(test_dataset)

(25000, 25000)

In [5]:
for i in range(4):
    review,label = train_dataset[i]
    print(review)
    print(label)   

An old saying goes "If you think you have problems, visit a hospital." That has been updated in recent years to "If you think you have problems, watch a TV talk show...especially Jerry Springer's!" This movie is one of those that is so bad, it's good! That's why I gave it a seven-it's all right, but not great. It's a great way to waste 95 minutes, just as the daily talk show is advertised as "an hour of your life you'll never get back!" All the familiar themes are here...unfaithful husbands/boyfriends, the wildest audience on television, women flashing Jerry, etc. The shocker was watching Molly Hagan, who normally plays sweet characters ("Seinfeld" and "Herman's Head") playing a trailer-trash mom and Jaime Pressly ("My Name Is Earl") as her equally trashy daughter, performing sexual favors for virtually every man with whom they came in contact. The men (including the staff producer) were presented as quintessential lunkheads who deserved what they got. I don't want to spoil or reveal e

Need to preprocess the text:
- Need to tokenize the text => break the text down into the units the model will think and process in
- Using a simple word tokenizer that uses whitespaces to break the text into distinct words - for this example

In [6]:
def tokenizer(text):
    # cleaning the text
    text = re.sub('<[^>]*>','', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = re.sub("[^\w']+",' ', text.lower()) + ' ' +' '.join(emoticons).replace('-','')
    # tokenizing the text
    tokenized = text.split()
    return tokenized

<>:4: SyntaxWarning: invalid escape sequence '\)'
<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\)'
<>:5: SyntaxWarning: invalid escape sequence '\w'
/var/folders/_h/yzv4_kzj2yv3z5xh7lks3hpr0000gn/T/ipykernel_1021/369664450.py:4: SyntaxWarning: invalid escape sequence '\)'
  emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
/var/folders/_h/yzv4_kzj2yv3z5xh7lks3hpr0000gn/T/ipykernel_1021/369664450.py:5: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub("[^\w']+",' ', text.lower()) + ' ' +' '.join(emoticons).replace('-','')


Testing it out.....

In [7]:
line, label = train_dataset[0]

In [8]:
line

'An old saying goes "If you think you have problems, visit a hospital." That has been updated in recent years to "If you think you have problems, watch a TV talk show...especially Jerry Springer\'s!" This movie is one of those that is so bad, it\'s good! That\'s why I gave it a seven-it\'s all right, but not great. It\'s a great way to waste 95 minutes, just as the daily talk show is advertised as "an hour of your life you\'ll never get back!" All the familiar themes are here...unfaithful husbands/boyfriends, the wildest audience on television, women flashing Jerry, etc. The shocker was watching Molly Hagan, who normally plays sweet characters ("Seinfeld" and "Herman\'s Head") playing a trailer-trash mom and Jaime Pressly ("My Name Is Earl") as her equally trashy daughter, performing sexual favors for virtually every man with whom they came in contact. The men (including the staff producer) were presented as quintessential lunkheads who deserved what they got. I don\'t want to spoil or

In [9]:
tokens = tokenizer(line)

In [10]:
tokens

['an',
 'old',
 'saying',
 'goes',
 'if',
 'you',
 'think',
 'you',
 'have',
 'problems',
 'visit',
 'a',
 'hospital',
 'that',
 'has',
 'been',
 'updated',
 'in',
 'recent',
 'years',
 'to',
 'if',
 'you',
 'think',
 'you',
 'have',
 'problems',
 'watch',
 'a',
 'tv',
 'talk',
 'show',
 'especially',
 'jerry',
 "springer's",
 'this',
 'movie',
 'is',
 'one',
 'of',
 'those',
 'that',
 'is',
 'so',
 'bad',
 "it's",
 'good',
 "that's",
 'why',
 'i',
 'gave',
 'it',
 'a',
 'seven',
 "it's",
 'all',
 'right',
 'but',
 'not',
 'great',
 "it's",
 'a',
 'great',
 'way',
 'to',
 'waste',
 '95',
 'minutes',
 'just',
 'as',
 'the',
 'daily',
 'talk',
 'show',
 'is',
 'advertised',
 'as',
 'an',
 'hour',
 'of',
 'your',
 'life',
 "you'll",
 'never',
 'get',
 'back',
 'all',
 'the',
 'familiar',
 'themes',
 'are',
 'here',
 'unfaithful',
 'husbands',
 'boyfriends',
 'the',
 'wildest',
 'audience',
 'on',
 'television',
 'women',
 'flashing',
 'jerry',
 'etc',
 'the',
 'shocker',
 'was',
 'watchin

The counter lib - gets the unique tokens and their frequencies.....

In [12]:
Counter(tokens)

Counter({'the': 8,
         'a': 6,
         'to': 5,
         'is': 5,
         "it's": 5,
         'you': 4,
         'in': 4,
         'as': 4,
         'that': 3,
         'show': 3,
         'jerry': 3,
         'movie': 3,
         'i': 3,
         'but': 3,
         'great': 3,
         'back': 3,
         'and': 3,
         'an': 2,
         'if': 2,
         'think': 2,
         'have': 2,
         'problems': 2,
         'talk': 2,
         'this': 2,
         'one': 2,
         'of': 2,
         'good': 2,
         'seven': 2,
         'all': 2,
         'not': 2,
         'daily': 2,
         'hour': 2,
         'here': 2,
         'who': 2,
         'plays': 2,
         'for': 2,
         'every': 2,
         'they': 2,
         'everything': 2,
         'old': 1,
         'saying': 1,
         'goes': 1,
         'visit': 1,
         'hospital': 1,
         'has': 1,
         'been': 1,
         'updated': 1,
         'recent': 1,
         'years': 1,
         'watch': 1,

Building the vocabulary using the tokenizer (the distinct tokens seen)

In [13]:
token_counts = Counter()

for line,label in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)

print('Vocab size: ', len(token_counts))

Vocab size:  88669


That's tokenization done and we'll now map each unique word to a unique integer (text vectorization). This can be done manually using a python dictionary, where the keys are the unique tokens (words) and the value associated with each key is a unique integer.

In [14]:
class Vocabulary:
    def __init__(self, ordered_dict:OrderedDict, unk_index: int = None):
        # ordered dict maps token -> frequency (frequency is ignored after init)
        self.itos: List[str] = list(ordered_dict.keys()) # index -> token
        self.stoi: Dict[str, int] = {tok:idx for idx,tok in enumerate(self.itos)}
        self.unk_index = unk_index
        self.default_index = unk_index
    
    def insert_token(self, token:str, index:int) -> None:
        """ Insert a token at a specific index (shifts existing entries)"""
        if token in self.stoi:
            # token already exists -> remove old entry and add new entry
            old_idx = self.stoi.pop(token)
            # shift everything after old_idx down
            for t, i in self.stoi.items():
                if i > old_idx:
                    self.stoi[t] = i-1
            self.itos = [t for t,_ in sorted(self.stoi.items(), key=lambda x:x[1])]
        
        # insert new token at the requested index
        self.itos.insert(index, token)
        self.stoi[token] = index
        # shift everything >= index by 1
        for t, i in self.stoi.items():
            if i>=index and t!=token:
                self.stoi[t] = i+1
    
    def set_default_index(self, idx: int) -> None:
        self.default_index = idx
    
    # convenience methods
    def __getitem__(self, token:str) -> int:
        return self.stoi.get(token, self.default_index)
    
    def __len__(self) -> int:
        return len(self.itos)
    
    def lookup_indices(self, tokens: List[str]) -> List[int]:
        return [self[t] for t in tokens]
    
    def lookup_tokens(self, indices:List[int])->List[str]:
        return [self.itos[i] for i in indices if i < len(self.itos)]

In [15]:
sorted_by_freq_tuples = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = Vocabulary(ordered_dict)
vocab.insert_token("<pad>",0)
vocab.insert_token("<unk>",1)
vocab.set_default_index(1)

Now, to demonstrate the working of the vocab object, we will convert an example input text into a list of integer values.

In [16]:
print([vocab[token] for token in ['this','is','an','example']])

[11, 7, 32, 457]


Now we've got a text tokenizer and a text vectorizer. (text to tokens, and tokens to integers.)

Any token not in the vocabulary will get assigned the index 1 - the unknown word/token index. Another reserved value is the integer 0, which serves as a placeholder, a so-called padding token, for adjusting the sequence length. We can now define the text pipeline function to transform each text accordingly.

In [17]:
text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]

We will generate batches of samples using DataLoader and pass the data processing pipelines declared previously to the argument collate_fn. We will wrap the text encoding into the collate_batch function.

Significance of the collate function argument of the DataLoader class:
- We're working with sequences of text of varying lengths as a result of the nature of the problem
- text classfication (we get varying length sequences that we need to correctly classify)
- We cannot be able to form batches the default way with this varying length data and for each batch, we need to ensure that all sequences are of the same length
- This is able to be done by padding short sequences with zeros to the max length of the longest sequences hence ensuring that all sequences in the same batch are of the same length
- This type of modification is possible using the collate_fn argument that takes a function that customises the way batching is done

In [18]:
def collate_batch(batch):
    # at this point - batch is a list of tuples composed of items from your Dataset class - i.e. a tuple of the output of your dataset class
    # this list of tuples is of length batch_size
    # these items are indexed iteratively from 0 to batch_size -1

    label_list, text_list, lengths = [], [], []

    # looping through that list of tuples of length batch size
    for _text, _label in batch:
        label_list.append(_label) # fetch each label and dump it in this list
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64) # tokenize and vectorise the text 
        text_list.append(processed_text) # and also dump the tensors in this list
        lengths.append(processed_text.size(0)) # get the lengths of the sequence of vectors and dump it in this list
    label_list = torch.tensor(label_list) # converting the label list to tensor form
    lengths = torch.tensor(lengths) # converting the lengths list to tensor form
    # nn.utils.rnn.pad_sequence => pads a list of variable length tensors with a pad value, returning a tensor
    padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True) # does the padding for us and subsequently batches everything up as well
    return padded_text_list, label_list, lengths

Testing out the collate function...

In [19]:
# take a small batch
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=collate_batch)

Illustrating and visualizing how this padding now works:

In [20]:
text_batch, label_batch, length_batch = next(iter(dataloader))

In [21]:
print(text_batch)

tensor([[   32,   151,   656,  ...,     0,     0,     0],
        [   11,    17,     7,  ...,     0,     0,     0],
        [    2,  2005,     7,  ...,     2,  5229,  1114],
        [  889,  1378, 30552,  ...,     0,     0,     0]])


In [22]:
print(label_batch)

tensor([1, 1, 1, 0])


In [23]:
print(length_batch)

tensor([224, 118, 422, 147])


In [24]:
print(text_batch.shape)

torch.Size([4, 422])


As seen from above, the padding ensured all sequences are of the length of the longest sequence for that batch.

So, dividing all 3 datasets into dataloaders with a batch size of 128...

In [25]:
# creating a validation dataset first.... random_split taking the dataset you want to split and the split sizes....
train_dataset, valid_dataset = random_split(train_dataset, [20000,5000])

In [26]:
batch_size = 128
train_dl = DataLoader(train_dataset, batch_size = batch_size, shuffle=True, collate_fn= collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

Now, the data is in a suitable format for an rnn model, which is implemented below:

Embedding layers for sentence encoding...
- Given a set of tokens of size n+2 (n is the size of the token set, plus index 0 reserved for the words not present in the token set), an embedding matrix of size (n+2)xembedding_dim will be created where each row of this matrix represents numeric features associated with a token. Therefore, when an integer index, i, is given as input to the embedding, it will lookup the corresponding row of the matrix at index i and return the numeric features. The embedding matrix serves as the input to our NN models. In practice, creating an embedding layer is simply done using the nn.Embedding module

In [27]:
embedding = nn.Embedding(
    num_embeddings = 10, # the number of unique integers to produce embeddings for...
    embedding_dim = 3, # embeddings dimensionality
    padding_idx = 0 # applies masking, prevents the embeddings related to this token id from being updated during training hence improving learning and performance
)

In [28]:
# a batch pf 2 samples of 4 indices each
text_encoded_input = torch.LongTensor([[1,2,4,5],[4,3,2,0]])
print(text_encoded_input.shape)

torch.Size([2, 4])


In [29]:
enc_embeddings = embedding(text_encoded_input)
print(enc_embeddings.shape)

torch.Size([2, 4, 3])


**Running into an issue below that I need to resolve here:**
- my custom rnn layer and cell not working for packed padded sequence data. Need these packed padded sequences to prevent the rnn (sequence model) from processing the padded tokens. need to resolve

---------- test and debug mode --------------------

In [30]:
# take a small batch
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=collate_batch)
t_data, t_labels, t_lengths = next(iter(dataloader))

In [31]:
t_data.shape, t_labels.shape, t_lengths.shape

(torch.Size([4, 343]), torch.Size([4]), torch.Size([4]))

In [32]:
# build out dummy embedding layer
d_embedding = nn.Embedding(len(vocab),20,padding_idx=0)

**Building a Custom RNN Cell with Layer Normalization.......:**
- Now building a custom rnn cell with layer normalization

In [55]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_hidden = nn.Linear(input_size, hidden_size)
        self.hidden_hidden = nn.Linear(hidden_size, hidden_size)
        # initializing the weights and biases
        nn.init.xavier_uniform_(self.input_hidden.weight)
        nn.init.zeros_(self.input_hidden.bias)

        nn.init.xavier_uniform_(self.hidden_hidden.weight)
        nn.init.zeros_(self.hidden_hidden.bias)

        # define the layer normalization layer
        self.layer_norm = nn.LayerNorm(hidden_size)
    
    def forward(self, x, h):
        computed_hidden = self.input_hidden(x) + self.hidden_hidden(h)
        computed_hidden = self.layer_norm(computed_hidden)
        return torch.tanh(computed_hidden)

In [56]:
class CustomRNNLayer(nn.Module):
    def __init__(self, input_size, hidden_size, device):
        super().__init__()
        self.rnn_cell = CustomRNNCell(input_size, hidden_size).to(device)
        self.hidden_size = hidden_size
        self.device = device
    
    def forward(self, packed):
        data, batch_sizes = packed.data, packed.batch_sizes
        max_batch = batch_sizes[0]
        h = torch.zeros(max_batch, self.hidden_size, device=self.device)

        outputs = []
        offset = 0

        for step, bs in enumerate(batch_sizes):
            x_t = data[offset:offset+bs]
            offset += bs

            h_t = self.rnn_cell(x_t, h[:bs])
            h = h.clone()
            h[:bs] = h_t
            outputs.append(h_t)
        new_data = torch.cat(outputs, dim=0)
        return nn.utils.rnn.PackedSequence(new_data, batch_sizes)

In [97]:
  
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, fc_hidden_size,device):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn_layer = CustomRNNLayer(embed_dim, hidden_size, device)
        self.fc1 = nn.Linear(hidden_size, fc_hidden_size)
        nn.init.kaiming_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, text, lengths):
         # text: (batch, seq_len)
        embedded = self.embedding(text)
        # embedded: (batch, seq_len, embed_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out = self.rnn_layer(packed)

        # UNPACK (critical)
        padded_out, out_lengths = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True
        )
        # padded_out: (batch, max_seq_len, hidden_size)

        # extract last valid hidden state
        batch_idx = torch.arange(padded_out.size(0), device=padded_out.device)
        last_hidden = padded_out[batch_idx, out_lengths - 1]
        # last_hidden: (batch, hidden_size)

        out = self.fc1(last_hidden)
        out = self.relu(out)
        out = self.fc2(out)
        return self.sigmoid(out)


In [98]:
vocab_size = len(vocab)
embed_dim = 20
hidden_size = 64
fc_hidden_size = 64
torch.manual_seed(1)

In [99]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [100]:
model = RNNModel(vocab_size, embed_dim, hidden_size, fc_hidden_size, device).to(device)

In [101]:
def train(dataloader):
    model.train()
    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:
        optimizer.zero_grad(set_to_none=True)
        label_batch = label_batch.float()
        text_batch, label_batch = text_batch.to(device), label_batch.to(device)
        pred = model(text_batch, lengths)[:,0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()
        total_acc += (
            (pred >= 0.5).float() == label_batch
        ).float().sum().item()
        total_loss += loss.item()*label_batch.size(0)
    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

In [102]:
def evaluate(dataloader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            label_batch = label_batch.float()
            text_batch, label_batch = text_batch.to(device), label_batch.to(device)
            pred = model(text_batch, lengths)[:,0]
            loss = loss_fn(pred, label_batch)
            total_acc += (
                (pred >= 0.5).float() == label_batch
            ).float().sum().item()
            total_loss += loss.item()*label_batch.size(0)
    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

In [103]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [104]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f'Epoch: {epoch} | accuracy: {acc_train:.4f} | val_accuracy: {acc_valid:.4f}')

Epoch: 0 | accuracy: 0.4953 | val_accuracy: 0.5032
Epoch: 1 | accuracy: 0.5065 | val_accuracy: 0.5050
Epoch: 2 | accuracy: 0.5003 | val_accuracy: 0.4960
Epoch: 3 | accuracy: 0.4988 | val_accuracy: 0.5054
Epoch: 4 | accuracy: 0.4943 | val_accuracy: 0.5120
Epoch: 5 | accuracy: 0.5027 | val_accuracy: 0.4988
Epoch: 6 | accuracy: 0.4993 | val_accuracy: 0.4960
Epoch: 7 | accuracy: 0.4960 | val_accuracy: 0.4978
Epoch: 8 | accuracy: 0.4954 | val_accuracy: 0.4918
Epoch: 9 | accuracy: 0.4961 | val_accuracy: 0.4902


In [107]:
lengs = torch.tensor([3,4,2])
torch.arange(4)[None,:] < lengs[:,None]

tensor([[ True,  True,  True, False],
        [ True,  True,  True,  True],
        [ True,  True, False, False]])

In [ ]:
torch.arange(4).shape

torch.Size([3])

In [104]:
torch.arange(3)[None,:].shape

torch.Size([1, 3])

In [102]:
torch.arange(3)[None,:]

tensor([[0, 1, 2]])

In [105]:
lengs.shape

torch.Size([3])

In [106]:
lengs[:,None].shape

torch.Size([3, 1])